# Victorian NEM Data Processing Pipeline

This notebook documents the processing workflow used to build the analytical dataset for the **Victorian Wholesale Electricity Price Analysis** portfolio project.

**Analysis period:** 1 August 2025 – 31 July 2026  
**Primary sources:** Australian Energy Market Operator (AEMO)

The pipeline combines:

- VIC1 five-minute Price & Demand data
- AEMO DISPATCHSCADA unit-level generation data
- AEMO Generation Information metadata

The final output is one row per five-minute settlement interval, containing price, demand, and Victorian generation by technology type.

> Note: AEMO's DISPATCHSCADA archive is rolling. Some early daily archives used during the original extraction may no longer be available later. The project therefore retains the processed generation dataset as a reproducible project input.

## 1. Setup and libraries

The raw DISPATCHSCADA files are nested ZIP archives: each daily archive contains approximately 288 five-minute ZIP files, each of which contains one CSV.

In [ ]:
import csv
import io
import zipfile
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# Repository-relative paths
DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GENERATOR_METADATA_FILE = DATA_DIR / "NEM_Generation_Information_July_2026.xlsx"
PRICE_DEMAND_DIR = DATA_DIR / "price_demand"

## 2. Load AEMO generator metadata

The AEMO Generation Information workbook contains multiple physical/configuration rows for some DUIDs.  
Before joining it to five-minute SCADA data, the metadata must be collapsed to **one row per DUID**.

In [ ]:
generator_meta = pd.read_excel(
    GENERATOR_METADATA_FILE,
    sheet_name="Generator Information",
    header=3
)

generator_lookup = generator_meta[
    [
        "Site Name",
        "Region",
        "Unit Name",
        "Technology Type",
        "Technology Detail",
        "Gas Turbine Fuel Type",
        "DUID",
        "Dispatch Type",
        "Aggregated Nameplate Capacity (MW AC)",
        "Commitment Status"
    ]
].copy()

generator_lookup = generator_lookup[
    generator_lookup["DUID"].notna()
].copy()

print("Metadata rows:", len(generator_lookup))
print("Unique DUIDs:", generator_lookup["DUID"].nunique())

### Validate metadata grain

A direct join at the raw metadata grain would duplicate SCADA rows for DUIDs represented by multiple physical units.  
The lookup is therefore aggregated to one row per DUID before joining.

In [ ]:
fields_to_check = [
    "Region",
    "Technology Type",
    "Technology Detail",
    "Dispatch Type",
    "Commitment Status",
    "Site Name"
]

duid_consistency = (
    generator_lookup
    .groupby("DUID")[fields_to_check]
    .nunique(dropna=False)
)

inconsistent_duids = duid_consistency[
    (duid_consistency > 1).any(axis=1)
]

print("DUIDs with inconsistent classification fields:")
display(inconsistent_duids)

In [ ]:
duid_lookup = (
    generator_lookup
    .groupby("DUID", as_index=False)
    .agg({
        "Site Name": "first",
        "Region": "first",
        "Technology Type": "first",
        "Technology Detail": "first",
        "Gas Turbine Fuel Type": "first",
        "Dispatch Type": "first",
        "Commitment Status": "first",
        "Aggregated Nameplate Capacity (MW AC)": "sum"
    })
)

assert len(duid_lookup) == duid_lookup["DUID"].nunique()
print("Clean DUID lookup rows:", len(duid_lookup))

## 3. Define the DISPATCHSCADA parser

The function below converts one daily AEMO DISPATCHSCADA ZIP into a clean Victorian generation-mix table:

1. Open the daily archive.
2. Parse each nested five-minute ZIP/CSV.
3. Convert the SCADA values and timestamps.
4. Join DUID metadata using a validated many-to-one relationship.
5. Keep VIC1 records.
6. Aggregate MW by technology and timestamp.
7. Pivot to one row per five-minute interval.

SCADAVALUE signs are preserved. This is important for battery storage because negative values can represent charging/load.

In [ ]:
def process_scada_day(daily_bytes, duid_lookup):
    all_frames = []

    with zipfile.ZipFile(io.BytesIO(daily_bytes), "r") as outer_zip:
        for nested_name in outer_zip.namelist():

            with outer_zip.open(nested_name) as nested_file:
                nested_bytes = nested_file.read()

            with zipfile.ZipFile(io.BytesIO(nested_bytes), "r") as nested_zip:
                csv_name = nested_zip.namelist()[0]

                with nested_zip.open(csv_name) as csv_file:
                    raw_text = csv_file.read().decode("utf-8")

            rows = list(csv.reader(io.StringIO(raw_text)))

            header_row = next(row for row in rows if row[0] == "I")
            columns = header_row[4:]

            data_rows = [
                row[4:]
                for row in rows
                if row[0] == "D"
            ]

            all_frames.append(
                pd.DataFrame(data_rows, columns=columns)
            )

    generation_day = pd.concat(all_frames, ignore_index=True)

    generation_day["SETTLEMENTDATE"] = pd.to_datetime(
        generation_day["SETTLEMENTDATE"]
    )
    generation_day["SCADAVALUE"] = pd.to_numeric(
        generation_day["SCADAVALUE"],
        errors="coerce"
    )

    generation_day = generation_day.merge(
        duid_lookup,
        on="DUID",
        how="left",
        validate="many_to_one"
    )

    vic_generation = generation_day[
        generation_day["Region"] == "VIC1"
    ].copy()

    generation_mix = (
        vic_generation
        .groupby(["SETTLEMENTDATE", "Technology Type"])["SCADAVALUE"]
        .sum()
        .reset_index()
    )

    generation_wide = (
        generation_mix
        .pivot(
            index="SETTLEMENTDATE",
            columns="Technology Type",
            values="SCADAVALUE"
        )
        .reset_index()
    )

    return generation_wide

## 4. Discover AEMO daily archive files

AEMO NEMWeb provides daily DISPATCHSCADA archives. The listing can be read programmatically instead of downloading hundreds of files manually.

Because the archive is rolling, the exact number of historical files visible today may differ from the number available during the original extraction.

In [ ]:
archive_url = "https://www.nemweb.com.au/REPORTS/ARCHIVE/Dispatch_SCADA/"
base_url = "https://www.nemweb.com.au"

response = requests.get(archive_url, timeout=60)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

zip_links = [
    link.get("href")
    for link in soup.find_all("a")
    if link.get("href", "").endswith(".zip")
]

print("ZIP files currently visible:", len(zip_links))

In [ ]:
from datetime import datetime

selected_files = []

for file_path in zip_links:
    filename = Path(file_path).name

    date_text = (
        filename
        .replace("PUBLIC_DISPATCHSCADA_", "")
        .replace(".zip", "")
    )

    file_date = datetime.strptime(date_text, "%Y%m%d")

    if datetime(2025, 8, 1) <= file_date <= datetime(2026, 7, 31):
        selected_files.append(file_path)

print("Files available for target period:", len(selected_files))
if selected_files:
    print("First:", selected_files[0])
    print("Last:", selected_files[-1])

## 5. Process daily SCADA archives

The following loop downloads each available daily archive into memory, processes it, and retains only the clean technology-level output.

For the published project, the fully processed 12-month generation dataset is retained in the repository/project storage so the later analysis is not dependent on AEMO's rolling archive retention.

In [ ]:
all_days = []
failed_files = []

for i, file_path in enumerate(selected_files, start=1):
    try:
        file_url = urljoin(base_url, file_path)
        print(f"[{i}/{len(selected_files)}] {Path(file_path).name}")

        response = requests.get(file_url, timeout=60)
        response.raise_for_status()

        daily_mix = process_scada_day(
            response.content,
            duid_lookup
        )

        daily_mix["Source File"] = Path(file_path).name
        all_days.append(daily_mix)

    except Exception as exc:
        failed_files.append({
            "file_path": file_path,
            "error": str(exc)
        })

print("Processed daily files:", len(all_days))
print("Failed files:", len(failed_files))

### Load the retained 12-month processed generation dataset

The original project extraction produced a complete 1 Aug 2025 – 31 Jul 2026 generation series except for 20 known five-minute SCADA gaps on 10 March 2026.

In [ ]:
generation_12months = pd.read_csv(
    DATA_DIR / "generation_mix_202508_to_202607.csv"
)

generation_12months["SETTLEMENTDATE"] = pd.to_datetime(
    generation_12months["SETTLEMENTDATE"]
)

print("Shape:", generation_12months.shape)
print("Unique timestamps:", generation_12months["SETTLEMENTDATE"].nunique())
print("Min:", generation_12months["SETTLEMENTDATE"].min())
print("Max:", generation_12months["SETTLEMENTDATE"].max())
print(
    "Duplicates:",
    generation_12months["SETTLEMENTDATE"].duplicated().sum()
)

## 6. Validate missing generation intervals

A complete 365-day five-minute time series contains **105,120 intervals**.  
The generation dataset contains **105,100 rows**, reflecting 20 missing source intervals. These are not imputed.

In [ ]:
expected_timestamps = pd.date_range(
    start="2025-08-01 00:05:00",
    end="2026-08-01 00:00:00",
    freq="5min"
)

actual_timestamps = pd.DatetimeIndex(
    generation_12months["SETTLEMENTDATE"]
)

missing_timestamps = expected_timestamps.difference(
    actual_timestamps
)

print("Missing generation timestamps:", len(missing_timestamps))
print(missing_timestamps)

## 7. Load 12 months of VIC Price & Demand data

In [ ]:
price_files = sorted(
    PRICE_DEMAND_DIR.glob("PRICE_AND_DEMAND_*_VIC1.csv")
)

print("Price & Demand files found:", len(price_files))

price_frames = [
    pd.read_csv(file)
    for file in price_files
]

price_demand_year = pd.concat(
    price_frames,
    ignore_index=True
)

price_demand_year["SETTLEMENTDATE"] = pd.to_datetime(
    price_demand_year["SETTLEMENTDATE"]
)
price_demand_year["TOTALDEMAND"] = pd.to_numeric(
    price_demand_year["TOTALDEMAND"]
)
price_demand_year["RRP"] = pd.to_numeric(
    price_demand_year["RRP"]
)

print("Shape:", price_demand_year.shape)
print("Unique timestamps:", price_demand_year["SETTLEMENTDATE"].nunique())
print("Min:", price_demand_year["SETTLEMENTDATE"].min())
print("Max:", price_demand_year["SETTLEMENTDATE"].max())
print(
    "Duplicate timestamps:",
    price_demand_year["SETTLEMENTDATE"].duplicated().sum()
)

## 8. Join Price, Demand and Generation

Price & Demand provides the complete master time spine.  
A left join preserves all 105,120 market intervals while leaving the 20 missing SCADA intervals explicitly null.

In [ ]:
final_data = price_demand_year.merge(
    generation_12months,
    on="SETTLEMENTDATE",
    how="left",
    validate="one_to_one"
)

print("Shape:", final_data.shape)
print(
    "Duplicate timestamps:",
    final_data["SETTLEMENTDATE"].duplicated().sum()
)
print(final_data.isna().sum())

## 9. Create analytical features

Battery Storage is deliberately kept separate because positive and negative SCADA values represent different operating modes.

In [ ]:
final_data["Generation Data Missing"] = final_data["Coal"].isna()

final_data["Renewable MW"] = (
    final_data["Wind"]
    + final_data["Solar PV"]
    + final_data["Hydro"]
)

final_data["Thermal MW"] = (
    final_data["Coal"]
    + final_data["Gas Turbine"]
)

final_data["Total Generation MW"] = (
    final_data["Renewable MW"]
    + final_data["Thermal MW"]
)

final_data["Renewable Share %"] = (
    final_data["Renewable MW"]
    / final_data["Total Generation MW"]
    * 100
)

final_data["Negative Price"] = final_data["RRP"] < 0

final_data["Demand Group"] = pd.qcut(
    final_data["TOTALDEMAND"],
    q=3,
    labels=["Low", "Medium", "High"]
)

final_data["Renewable Group"] = pd.qcut(
    final_data["Renewable MW"],
    q=3,
    labels=["Low", "Medium", "High"]
)

final_data["Solar Group"] = pd.qcut(
    final_data["Solar PV"],
    q=3,
    labels=["Low", "Medium", "High"],
    duplicates="drop"
)

final_data["Wind Group"] = pd.qcut(
    final_data["Wind"],
    q=3,
    labels=["Low", "Medium", "High"],
    duplicates="drop"
)

## 10. Final validation

In [ ]:
print("Final shape:", final_data.shape)
print(
    "Unique timestamps:",
    final_data["SETTLEMENTDATE"].nunique()
)
print(
    "Duplicate timestamps:",
    final_data["SETTLEMENTDATE"].duplicated().sum()
)
print(
    "Generation gaps:",
    final_data["Generation Data Missing"].sum()
)

assert len(final_data) == 105_120
assert final_data["SETTLEMENTDATE"].nunique() == 105_120
assert final_data["SETTLEMENTDATE"].duplicated().sum() == 0
assert final_data["Generation Data Missing"].sum() == 20

## 11. Export final analytical dataset

This CSV is the source used by the Power BI dashboard and the analysis notebook.

In [ ]:
output_file = OUTPUT_DIR / "final_market_dataset.csv"

final_data.to_csv(
    output_file,
    index=False
)

print("Saved:", output_file)

## Data quality notes

- The final Price & Demand time spine contains 105,120 five-minute intervals.
- 20 generation intervals on 10 March 2026 are absent from the AEMO DISPATCHSCADA source and are retained as missing rather than imputed.
- Raw AEMO generator metadata contains repeated DUIDs at a finer physical/configuration grain; the metadata was collapsed to one DUID-level lookup before joining.
- SCADAVALUE signs are preserved, especially for battery charging/discharging behaviour.
- Relationships analysed later are associations and should not be interpreted as proof of causation.